In [ ]:
"""
Aesthetic Tile-Based Map
Optimized for 9:16 portrait format (1080x1920) - ideal for mobile viewing and social media
"""

import os
import pandas as pd
import geopandas as gpd
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
from pathlib import Path
from PIL import Image
import numpy as np
import time
import contextily as ctx

# === CONFIGURATION =======================================================
# Use sample_data for testing, or data/mapillary for your own data
csv_path = Path("sample_data") / "metadata.csv"
image_folder = Path("sample_data")

# For your own data, uncomment these lines:
# csv_path = Path("data") / "mapillary" / "metadata.csv"
# image_folder = Path("data") / "mapillary"

target_user = "mapfool"
gif_output = Path("PORTRAIT_9x16.gif")
duration = 1.5
max_frames = 15
max_total_points = 50

# Karakoy polygon
polygon_coords = [
    (28.97201188246599, 41.027961514127306),
    (28.972796376763487, 41.02567058090895),
    (28.97595966022113, 41.02547966620972),
    (28.982589902348348, 41.028171512351044),
    (28.986841355315423, 41.02924058383375),
    (28.98772707468357, 41.02939330691432),
    (28.98954912595517, 41.02809514943808),
    (28.99051076412629, 41.02637696048352),
    (28.98522175418511, 41.02311227795534),
    (28.9803376445265, 41.02127940279426),
    (28.97727558613951, 41.02040113203624),
    (28.976187416630072, 41.02074480459719),
    (28.975200472191293, 41.02124121735267),
    (28.973783321202273, 41.02143214433928),
    (28.971784126057038, 41.02156579290059),
    (28.96986084971479, 41.02210038443376),
    (28.968443698725757, 41.02315046231212),
    (28.967684510695936, 41.02404778832296),
    (28.967279610413353, 41.02460145146428),
    (28.967380835484, 41.02536511712449),
    (28.968038798443192, 41.02584240366503),
    (28.96869676140238, 41.02668241957435),
    (28.96889921154367, 41.02694969511649),
    (28.97201188246599, 41.027961514127306)
]
polygon = Polygon(polygon_coords)
# =========================================================================

print("=" * 60)
print("GIF GENERATION (9:16)")
print("=" * 60)

# Load CSV
print("\n[1/4] Loading CSV...")
t0 = time.time()
df = pd.read_csv(csv_path)
print(f"      Loaded {len(df)} rows in {time.time()-t0:.2f}s")

# Create GeoDataFrame
print("\n[2/4] Creating GeoDataFrame...")
t0 = time.time()
gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.lon, df.lat)],
    crs="EPSG:4326"
)
print(f"      Created in {time.time()-t0:.2f}s")

# Filter by user and polygon
print(f"\n[3/4] Filtering (user='{target_user}', polygon)...")
t0 = time.time()
gdf_user = gdf[(gdf['username'] == target_user) & (gdf.geometry.within(polygon))].copy()
print(f"      Found {len(gdf_user)} points")

# Convert to Web Mercator for proper distance calculation
print("\n[4/4] Sorting by nearest neighbor (meter-based distance)...")
t0 = time.time()
gdf_user_metric = gdf_user.to_crs(epsg=3857)

def sort_by_nearest_neighbor(gdf_metric, gdf_original, max_points):
    """
    Sort by nearest neighbor using REAL distance in meters (Web Mercator).
    Stops once max_points are collected to keep runtime low.
    """
    if len(gdf_metric) == 0:
        return gdf_original

    coords = np.vstack((gdf_metric.geometry.x.values, gdf_metric.geometry.y.values)).T
    idx_lookup = gdf_metric.index.to_numpy()
    n = len(coords)

    visited = np.zeros(n, dtype=bool)
    ordered_indices = []

    current_idx = coords[:, 0].argmin()  # westernmost point

    print(f"      Sorting up to {min(max_points, n)} of {n} points...")

    for _ in range(n):
        ordered_indices.append(idx_lookup[current_idx])
        visited[current_idx] = True

        if len(ordered_indices) >= max_points or visited.all():
            break

        remaining = np.where(~visited)[0]
        diffs = coords[remaining] - coords[current_idx]
        dist2 = (diffs * diffs).sum(axis=1)
        current_idx = remaining[dist2.argmin()]

    return gdf_original.loc[ordered_indices].reset_index(drop=True)

gdf_user = sort_by_nearest_neighbor(gdf_user_metric, gdf_user, max_frames)

gdf_user = gdf_user.iloc[:max_frames]
print(f"      Sorted {len(gdf_user)} points in {time.time()-t0:.2f}s")

# Convert to Web Mercator for display
gdf_user_web = gdf_user.to_crs(epsg=3857)
# Preserve map extent using full polygon area (not just traveled points)
poly_bounds_3857 = gpd.GeoSeries([polygon], crs="EPSG:4326").to_crs(epsg=3857).total_bounds
pad_x = (poly_bounds_3857[2] - poly_bounds_3857[0]) * 0.05
pad_y = (poly_bounds_3857[3] - poly_bounds_3857[1]) * 0.05
map_extent = (
    poly_bounds_3857[0] - pad_x,
    poly_bounds_3857[2] + pad_x,
    poly_bounds_3857[1] - pad_y,
    poly_bounds_3857[3] + pad_y,
)



# Generate frames
print(f"\nGenerating {len(gdf_user)} portrait frames...")
t0 = time.time()

frames = []
for i, (idx, row) in enumerate(gdf_user_web.iterrows(), 1):
    t_frame = time.time()
    
    fig = plt.figure(figsize=(10.8, 19.2), dpi=100)
    fig.patch.set_facecolor('white')
    
    ax_map = plt.subplot(2, 1, 1)
    
    visited = gdf_user_web.loc[:idx]
    if len(visited) > 1:
        for j in range(len(visited) - 1):
            p1 = visited.iloc[j].geometry
            p2 = visited.iloc[j + 1].geometry
            alpha_val = 0.3 + (j / len(visited)) * 0.5
            color = plt.cm.viridis(j / len(visited))
            ax_map.plot([p1.x, p2.x], [p1.y, p2.y],
                       color=color, linewidth=6, alpha=alpha_val,
                       solid_capstyle='round', zorder=2)
    
    gdf_user_web.plot(ax=ax_map, color="#3498db", markersize=60,
                     alpha=0.2, zorder=3, edgecolor='white', linewidth=1)
    
    if len(visited) > 0:
        visited.plot(ax=ax_map, color="#2ecc71", markersize=90,
                    alpha=0.5, zorder=4, edgecolor='white', linewidth=1.5)
    
    current_point = gpd.GeoSeries([row.geometry], crs="EPSG:3857")
    current_point.plot(ax=ax_map, color="#e74c3c", markersize=350,
                      alpha=1.0, zorder=5, edgecolor='white', linewidth=4)
    current_point.plot(ax=ax_map, color="#e74c3c", markersize=550,
                      alpha=0.3, zorder=4, edgecolor='none')
    
    if map_extent:
        ax_map.set_xlim(map_extent[0], map_extent[1])
        ax_map.set_ylim(map_extent[2], map_extent[3])

    try:
        ctx.add_basemap(ax_map, source=ctx.providers.CartoDB.Positron,
                       zoom=16, attribution="", alpha=0.9)
    except Exception as e:
        print(f"      Basemap failed: {e}")

    ax_map.set_title(f"Karakoy Route - {i}/{len(gdf_user)}",
                    fontsize=24, fontweight='bold', pad=20, color='#2c3e50')
    ax_map.set_xlabel("")
    ax_map.set_ylabel("")
    ax_map.tick_params(labelsize=10, colors='#7f8c8d')
    ax_map.grid(False)
    ax_map.set_aspect("equal")
    
    for spine in ax_map.spines.values():
        spine.set_edgecolor('#bdc3c7')
        spine.set_linewidth(2)
    
    ax_img = plt.subplot(2, 1, 2)
    original_row = gdf_user.loc[idx]
    img_path = image_folder / original_row['filename']
    
    if img_path.exists():
        try:
            img = Image.open(img_path)
            ax_img.imshow(img)
            coord_text = f"{original_row['lat']:.5f}, {original_row['lon']:.5f}"
            ax_img.text(0.5, -0.03, coord_text, transform=ax_img.transAxes,
                       ha='center', va='top', fontsize=16, color='white',
                       weight='bold', bbox=dict(boxstyle='round,pad=0.7',
                       facecolor='#2c3e50', edgecolor='none', alpha=0.85))
        except:
            ax_img.text(0.5, 0.5, "Image Error", ha='center', va='center',
                       fontsize=18, color='#e74c3c', weight='bold')
    else:
        ax_img.text(0.5, 0.5, "Image Not Found", ha='center', va='center',
                   fontsize=20, color='#e74c3c', weight='bold')
    
    ax_img.axis('off')
    plt.tight_layout(pad=1.0)
    
    fig.canvas.draw()
    frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    frames.append(frame)
    plt.close(fig)
    
    print(f"      Frame {i}/{len(gdf_user)} done ({frame.shape[1]}x{frame.shape[0]}px) - {time.time()-t_frame:.2f}s")

print(f"      All frames in {time.time()-t0:.2f}s")

# Save GIF
print(f"\nSaving 9:16 portrait GIF...")
t0 = time.time()

if len(frames) >= 2:
    imageio.mimsave(str(gif_output), frames, duration=duration, loop=0)
    file_size = gif_output.stat().st_size / 1024 / 1024
    print(f"\n{'='*60}")
    print(f"SUCCESS!")
    print(f"{'='*60}")
    print(f"   File: {gif_output.name}")
    print(f"   Size: {frames[0].shape[1]}x{frames[0].shape[0]}px (9:16)")
    print(f"   Frames: {len(frames)}")
    print(f"   Duration: {len(frames) * duration:.1f}s")
    print(f"   File Size: {file_size:.2f} MB")
    print(f"   Generated in {time.time()-t0:.2f}s")
    print(f"{'='*60}")
else:
    print("Not enough frames")

if frames:
    print("\nShowing preview...")
    plt.figure(figsize=(9, 16))
    plt.imshow(frames[0])
    plt.title("Preview (9:16)", fontsize=16, weight='bold', pad=15)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

print("\n9:16 Portrait GIF ready!")

# 9:16 PORTRAIT FORMAT - Aesthetic Map

**What it does:** Vertical (portrait) GIF optimized for mobile viewing and social media

**Features:**
- 9:16 aspect ratio (1080x1920 optimal for mobile)
- Vertical layout (perfect for mobile viewing)
- All Aesthetic Map features:
  - Real map tiles (CartoDB)
  - Gradient path + glow effect
  - Modern colors
- Mobile-optimized: Large fonts, clear view
- Ready for presentations and social media

**Output size:** 1080x1920 pixels (9:16 portrait standard)

In [ ]:
"""
FAST & SMART VERSION - Synchronized Map + Street View Animation
Greedy Nearest Neighbor with CORRECT meter-based distance calculation
"""

import os
import pandas as pd
import geopandas as gpd
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import osmnx as ox
from pathlib import Path
from PIL import Image
import numpy as np
import time

# === CONFIGURATION =======================================================
# Use sample_data for testing, or data/mapillary for your own data
csv_path = Path("sample_data") / "metadata.csv"
image_folder = Path("sample_data")

# For your own data, uncomment these lines:
# csv_path = Path("data") / "mapillary" / "metadata.csv"
# image_folder = Path("data") / "mapillary"

target_user = "mapfool"
gif_output = Path("map_street_SMART.gif")
duration = 1.5
max_frames = 15
max_total_points = 50

# Karakoy polygon
polygon_coords = [
    (28.97201188246599, 41.027961514127306),
    (28.972796376763487, 41.02567058090895),
    (28.97595966022113, 41.02547966620972),
    (28.982589902348348, 41.028171512351044),
    (28.986841355315423, 41.02924058383375),
    (28.98772707468357, 41.02939330691432),
    (28.98954912595517, 41.02809514943808),
    (28.99051076412629, 41.02637696048352),
    (28.98522175418511, 41.02311227795534),
    (28.9803376445265, 41.02127940279426),
    (28.97727558613951, 41.02040113203624),
    (28.976187416630072, 41.02074480459719),
    (28.975200472191293, 41.02124121735267),
    (28.973783321202273, 41.02143214433928),
    (28.971784126057038, 41.02156579290059),
    (28.96986084971479, 41.02210038443376),
    (28.968443698725757, 41.02315046231212),
    (28.967684510695936, 41.02404778832296),
    (28.967279610413353, 41.02460145146428),
    (28.967380835484, 41.02536511712449),
    (28.968038798443192, 41.02584240366503),
    (28.96869676140238, 41.02668241957435),
    (28.96889921154367, 41.02694969511649),
    (28.97201188246599, 41.027961514127306)
]
polygon = Polygon(polygon_coords)
# =========================================================================

print("=" * 60)
print("FAST & SMART GIF GENERATION")
print("=" * 60)

print("\n[1/5] Loading CSV...")
t0 = time.time()
df = pd.read_csv(csv_path)
print(f"      Loaded {len(df)} rows in {time.time()-t0:.2f}s")

print("\n[2/5] Creating GeoDataFrame...")
t0 = time.time()
gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.lon, df.lat)],
    crs="EPSG:4326"
)
print(f"      Created in {time.time()-t0:.2f}s")

print(f"\n[3/5] Filtering (user='{target_user}', polygon)...")
t0 = time.time()
gdf_user = gdf[(gdf['username'] == target_user) & (gdf.geometry.within(polygon))].copy()
print(f"      Found {len(gdf_user)} points")

print("\n[4/5] Smart sorting with METER-BASED distance...")
t0 = time.time()
gdf_user_metric = gdf_user.to_crs(epsg=3857)

def sort_by_nearest_neighbor(gdf_metric, gdf_original, max_points):
    """
    Sort by nearest neighbor using REAL distance in meters (Web Mercator).
    Stops once max_points are collected to keep runtime low.
    """
    if len(gdf_metric) == 0:
        return gdf_original

    coords = np.vstack((gdf_metric.geometry.x.values, gdf_metric.geometry.y.values)).T
    idx_lookup = gdf_metric.index.to_numpy()
    n = len(coords)

    visited = np.zeros(n, dtype=bool)
    ordered_indices = []

    current_idx = coords[:, 0].argmin()  # westernmost point

    print(f"      Sorting up to {min(max_points, n)} of {n} points...")

    for _ in range(n):
        ordered_indices.append(idx_lookup[current_idx])
        visited[current_idx] = True

        if len(ordered_indices) >= max_points or visited.all():
            break

        remaining = np.where(~visited)[0]
        diffs = coords[remaining] - coords[current_idx]
        dist2 = (diffs * diffs).sum(axis=1)
        current_idx = remaining[dist2.argmin()]

    return gdf_original.loc[ordered_indices].reset_index(drop=True)

gdf_user = sort_by_nearest_neighbor(gdf_user_metric, gdf_user, max_frames)

gdf_user = gdf_user.iloc[:max_frames]
print(f"      Sorted {len(gdf_user)} points in {time.time()-t0:.2f}s")
# Preserve map extent using full polygon area (not just traveled points)
poly_bounds = gpd.GeoSeries([polygon], crs="EPSG:4326").total_bounds
pad_x = (poly_bounds[2] - poly_bounds[0]) * 0.05
pad_y = (poly_bounds[3] - poly_bounds[1]) * 0.05
map_extent = (
    poly_bounds[0] - pad_x,
    poly_bounds[2] + pad_x,
    poly_bounds[1] - pad_y,
    poly_bounds[3] + pad_y,
)


print(f"\n[5/5] Fetching OSM data (may take 5-15s)...")
t0 = time.time()

try:
    buildings = ox.features_from_polygon(polygon, {"building": True})
    print(f"      Buildings: {len(buildings)}")
except:
    buildings = gpd.GeoDataFrame()
    print(f"      Buildings failed")

try:
    roads = ox.features_from_polygon(polygon, {"highway": True})
    print(f"      Roads: {len(roads)}")
except:
    roads = gpd.GeoDataFrame()
    print(f"      Roads failed")

print(f"      OSM data done in {time.time()-t0:.2f}s")

print(f"\nGenerating {len(gdf_user)} frames...")
t0 = time.time()

frames = []
for i, (idx, row) in enumerate(gdf_user.iterrows(), 1):
    t_frame = time.time()
    
    fig = plt.figure(figsize=(10, 13), dpi=80)
    ax_map = plt.subplot(2, 1, 1)
    
    if not buildings.empty:
        buildings.plot(ax=ax_map, facecolor="#eeeeee", edgecolor="none")
    if not roads.empty:
        roads.plot(ax=ax_map, color="lightgray", linewidth=0.4)
    
    gdf_user.plot(ax=ax_map, color="skyblue", markersize=20, alpha=0.3, zorder=2)
    gpd.GeoSeries([row.geometry], crs="EPSG:4326").plot(
        ax=ax_map, color="crimson", markersize=150, alpha=0.9, zorder=3,
        edgecolor="white", linewidth=1.5
    )
    
    visited = gdf_user.loc[:idx]
    if len(visited) > 1:
        line_coords = [(p.x, p.y) for p in visited.geometry]
        lx, ly = zip(*line_coords)
        ax_map.plot(lx, ly, color='blue', linewidth=1.5, alpha=0.6, zorder=1)
    
    if map_extent:
        ax_map.set_xlim(map_extent[0], map_extent[1])
        ax_map.set_ylim(map_extent[2], map_extent[3])

    ax_map.set_title(f"Frame {i}/{len(gdf_user)} (Smart Route)", fontsize=14, fontweight='bold')
    ax_map.set_xlabel("Longitude", fontsize=10)
    ax_map.set_ylabel("Latitude", fontsize=10)
    ax_map.grid(True, alpha=0.2)
    ax_map.set_aspect("equal")
    
    ax_img = plt.subplot(2, 1, 2)
    img_path = image_folder / row['filename']
    if img_path.exists():
        try:
            img = Image.open(img_path)
            ax_img.imshow(img)
            ax_img.set_title(f"({row['lat']:.5f}, {row['lon']:.5f})", fontsize=11)
        except:
            ax_img.text(0.5, 0.5, "Image error", ha='center', va='center')
    else:
        ax_img.text(0.5, 0.5, "Not found", ha='center', va='center')
    ax_img.axis('off')
    
    plt.tight_layout(pad=1.5)
    
    fig.canvas.draw()
    frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    frames.append(frame)
    plt.close(fig)
    
    print(f"      Frame {i}/{len(gdf_user)} done in {time.time()-t_frame:.2f}s")

print(f"      All frames in {time.time()-t0:.2f}s")

print(f"\nSaving GIF...")
t0 = time.time()

if len(frames) >= 2:
    imageio.mimsave(str(gif_output), frames, duration=duration, loop=0)
    file_size = gif_output.stat().st_size / 1024 / 1024
    print(f"\n{'='*60}")
    print(f"SUCCESS!")
    print(f"{'='*60}")
    print(f"   File: {gif_output.name}")
    print(f"   Frames: {len(frames)}")
    print(f"   Duration: {len(frames) * duration:.1f}s")
    print(f"   Size: {file_size:.2f} MB")
    print(f"   Saved in {time.time()-t0:.2f}s")
    print(f"   Method: Smart Nearest Neighbor (METER-BASED)")
else:
    print("Not enough frames")

if frames:
    plt.figure(figsize=(10, 13))
    plt.imshow(frames[0])
    plt.title("First Frame Preview (Smart Route)", fontsize=12)
    plt.axis('off')
    plt.show()

# SMART VERSION - Synchronized Map + Street

**What it does:** Best combination - FAST + SMOOTH route

**Features:**
- **Greedy Nearest Neighbor** (go to nearest point from each point)
- **FAST**: ~0.1 seconds for 15 points
- **SMOOTH**: Not random, logical ordered route
- **Best choice:** Both fast and beautiful

**When to use:** Almost always - most balanced option

In [ ]:
"""
Route-Based Animation (Alternative Settings)
Creates a GIF along a defined route with different speed/quality settings.

Use case: Quick preview or different animation speed
"""

import os
import pandas as pd
import imageio.v2 as imageio
from shapely.geometry import LineString, Point
import matplotlib.pyplot as plt
from pathlib import Path
import time

# === CONFIGURATION ========================================================
# Use sample_data for testing, or data/mapillary for your own data
csv_path = Path("sample_data") / "metadata.csv"
image_folder = Path("sample_data")

# For your own data, uncomment these lines:
# csv_path = Path("data") / "mapillary" / "metadata.csv"
# image_folder = Path("data") / "mapillary"

# Define your route (start and end points)
start_point = (28.970, 41.021)        # Western endpoint (lon, lat)
end_point   = (28.988, 41.027)        # Eastern endpoint (lon, lat)
radius_m    = 50                      # Buffer distance in meters

# Output settings
gif_output = Path("route_fast.gif")
duration = 0.1                        # Faster animation (0.1 seconds per frame)
# =========================================================================

print("=" * 70)
print("FAST ROUTE ANIMATION (Alternative Settings)")
print("=" * 70)

# === LOAD DATA ===========================================================
print("\n[1/5] Loading CSV file...")
t0 = time.time()
df = pd.read_csv(csv_path)
print(f"      Loaded {len(df)} rows ({time.time()-t0:.2f}s)")

# === CREATE ROUTE GEOMETRY ===============================================
print("\n[2/5] Creating route geometry...")
print(f"      Start: {start_point}")
print(f"      End: {end_point}")
print(f"      Buffer distance: {radius_m}m (WIDE)")

t0 = time.time()
line = LineString([start_point, end_point])
buf_deg = radius_m / 111_139
buffer_geom = line.buffer(buf_deg)
print(f"      Route ready ({time.time()-t0:.2f}s)")

# === FILTER AND ORDER IMAGES =============================================
print(f"\n[3/5] Filtering images along route...")
t0 = time.time()

frames_with_distance = []
processed = 0
filtered = 0

for _, row in df.iterrows():
    processed += 1
    try:
        pt = Point(row['lon'], row['lat'])
        
        if not buffer_geom.contains(pt):
            continue
        
        filtered += 1
        s = line.project(pt)
        img_path = image_folder / row['filename']
        
        if img_path.exists():
            frames_with_distance.append((s, str(img_path)))
        
        if processed % 1000 == 0:
            print(f"      Processed: {processed}/{len(df)} rows, {filtered} on route...")
        
    except Exception as e:
        continue

print(f"      Filtering complete ({time.time()-t0:.2f}s)")
print(f"      Total processed: {processed} rows")
print(f"      Inside buffer: {filtered} points")
print(f"      Available images: {len(frames_with_distance)} files")

# === SORT BY PROJECTION DISTANCE =========================================
print(f"\n[4/5] Sorting images along route...")
t0 = time.time()
frames_with_distance.sort(key=lambda x: x[0])
print(f"      Sorting complete ({time.time()-t0:.2f}s)")

# === LOAD IMAGES =========================================================
print(f"\n[5/5] Loading images...")
t0 = time.time()

ordered_images = []
for i, (_, img_path) in enumerate(frames_with_distance, 1):
    try:
        img = imageio.imread(img_path)
        ordered_images.append(img)
        
        if i % 20 == 0:
            print(f"      {i}/{len(frames_with_distance)} images loaded...")
    except Exception as e:
        print(f"      Error: {img_path} | {e}")

print(f"      {len(ordered_images)} images loaded ({time.time()-t0:.2f}s)")

# === CREATE GIF ==========================================================
print(f"\nCreating GIF...")
t0 = time.time()

if len(ordered_images) >= 2:
    imageio.mimsave(str(gif_output), ordered_images, duration=duration, loop=0)
    file_size = gif_output.stat().st_size / 1024 / 1024
    
    print("\n" + "=" * 70)
    print("SUCCESS!")
    print("=" * 70)
    print(f"   File: {gif_output}")
    print(f"   Frame count: {len(ordered_images)}")
    print(f"   Total duration: {len(ordered_images) * duration:.1f} seconds")
    print(f"   File size: {file_size:.2f} MB")
    print(f"   Save time: {time.time()-t0:.2f}s")
    print(f"   Animation speed: FAST (0.1s/frame)")
    print("=" * 70)
else:
    print("Not enough frames, GIF creation failed.")

# === DISPLAY FIRST FRAME =================================================
if ordered_images:
    plt.figure(figsize=(10, 10))
    plt.imshow(ordered_images[0])
    plt.title(f"Route Animation (first frame) - {len(ordered_images)} frames", fontsize=10)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# Fast Route Animation (Alternative Settings)

**What it does:** Different speed/quality version of first route animation

**Differences:**
- Wider buffer (50m)
- Faster animation (0.1s/frame)

**Use case:** Quick preview or timelapse effect